# Autograd 与反向传播

## 学习目标

能够追踪标量损失的计算图，解释叶子张量、梯度累积、`detach` 和禁用梯度。


## 概念模型与执行路径

前向运算构建动态图，反向传播沿图应用链式法则。梯度只自动累积到需要梯度的叶子张量；优化器更新前必须清理旧梯度。


### 实验 1：构建计算图并求叶子张量的梯度

**实验目的**：观察一次完整的自动微分过程：前向运算生成预测值和标量损失，`backward()` 再把损失对参数 `w` 的梯度累积到 `w.grad`。

这里定义 $p=\sum_i x_iw_i$、$L=(p-4)^2$。代入 `x=[1, 2, 3]` 和 `w=[0.5, 0.5, 0.5]`，得到 $p=3$、$L=1$。根据链式法则：

$$\frac{\partial L}{\partial w_i}=2(p-4)x_i=-2x_i$$

所以预期梯度为 `[-2, -4, -6]`。`w` 是由用户直接创建、设置了 `requires_grad=True` 的**叶子张量**，因此 PyTorch 会把梯度写入 `w.grad`；`prediction` 和 `loss` 是运算产生的非叶子张量。`loss.grad_fn` 显示损失由幂运算创建，说明它仍连接在动态计算图上。

**观察重点**：`backward()` 要从标量 `loss` 出发，沿 `PowBackward0 -> SumBackward -> MulBackward` 的方向应用链式法则。打印出的对象地址每次运行可能不同，不影响计算结果。首次单独运行本实验前，`w.grad` 应为 `None`。

In [4]:
import torch
x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
prediction = (x * w).sum()
loss = (prediction - 4.0) ** 2
print("prediction:", prediction.item(), "loss:", loss.item(), "grad_fn:", loss.grad_fn)
loss.backward()
print("d(loss)/d(w):", w.grad)


prediction: 3.0 loss: 1.0 grad_fn: <PowBackward0 object at 0x11c1f0df0>
d(loss)/d(w): tensor([-2., -4., -6.])


### 实验 2：验证梯度累积并清空梯度

**实验目的**：证明 `.backward()` 默认执行的是梯度累加，而不是覆盖。实验先复制实验 1 得到的梯度，再重新执行相同的前向表达式并反向传播一次，因此 `w.grad` 从 `[-2, -4, -6]` 变为 `[-4, -8, -12]`。断言用于把这个预期变成可执行检查。

必须重新计算损失，是因为实验 1 的 `loss.backward()` 已默认释放该图为反向传播保存的中间值；直接再次调用 `loss.backward()` 会报“Trying to backward through the graph a second time”。只有确实需要对同一张图反向多次时才考虑 `retain_graph=True`，因为保留图会增加内存占用。

最后把 `w.grad` 设为 `None`，为后续计算清除历史梯度。训练循环中通常调用 `optimizer.zero_grad(set_to_none=True)`；与填零相比，设为 `None` 往往能减少写内存，并让“尚未产生梯度”和“梯度恰好为零”保持可区分。

**执行依赖**：本实验使用实验 1 创建的 `x`、`w` 和 `w.grad`，应按顺序运行。

In [2]:
first_gradient = w.grad.clone()
((x * w).sum() - 4.0).pow(2).backward()
print("gradient accumulated:", w.grad)
torch.testing.assert_close(w.grad, first_gradient * 2)
w.grad = None


gradient accumulated: tensor([ -4.,  -8., -12.])


### 实验 3：比较 `no_grad` 与 `detach`

**实验目的**：比较两种切断自动微分记录的方式。`torch.no_grad()` 是一个作用域开关：作用域内执行的运算不进入计算图，适合模型推理或不需要梯度的参数更新。`prediction.detach()` 则返回一个与原张量共享底层存储、但从当前计算图分离的新张量。

因此，`inference_value.requires_grad` 和 `detached.requires_grad` 都是 `False`，但两者的时机不同：前者从运算发生时就不记录图，后者是在已有的 `prediction` 上创建无梯度视图。`detach()` 通常不会复制数据；如果还要原地修改分离后的值，又不希望影响原张量，应使用 `prediction.detach().clone()`。

**观察重点**：离开 `no_grad` 作用域后，自动微分会恢复。`w.requires_grad` 本身没有被改变，只是该作用域内由它计算出的结果不跟踪梯度。

In [4]:
with torch.no_grad():
    inference_value = (x * w).sum()
detached = prediction.detach()
print("no_grad requires_grad:", inference_value.requires_grad)
print("detached requires_grad:", detached.requires_grad)


no_grad requires_grad: False
detached requires_grad: False


### 实验 4：使用 `torch.autograd.grad` 直接取得梯度

**实验目的**：展示不依赖参数 `.grad` 缓冲区的函数式求导。实验创建新的叶子张量 `w2` 和损失 `loss2`，然后通过 `torch.autograd.grad(loss2, w2)` 明确指定“输出对哪个输入求导”。返回值是元组，因为该 API 支持同时对多个输入求梯度。

结果仍为 `[-2, -4, -6]`，与实验 1 的手算和 `backward()` 一致。关键区别是：`autograd.grad` 直接返回梯度，不会把结果累积到 `w2.grad`，所以调用后 `w2.grad` 仍为 `None`。它适合梯度惩罚、元学习以及只需要局部导数的场景；常规模型训练通常使用 `loss.backward()`，让所有叶子参数的 `.grad` 供优化器统一读取。

**注意**：该调用默认也会释放计算图。若要基于这个梯度继续求二阶导数，需要设置 `create_graph=True`。

In [4]:
w2 = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
loss2 = ((x * w2).sum() - 4.0) ** 2
analytical = torch.autograd.grad(loss2, w2)[0]
print("autograd.grad:", analytical)


autograd.grad: tensor([-2., -4., -6.])


## 底层机制

PyTorch 的 Autograd 在前向阶段为参与求导的运算建立动态计算图。图节点的 `grad_fn` 知道反向传播时怎样把上游梯度转换成对各输入的梯度；部分节点还会保存反向公式所需的前向中间值。

因此，把仍连接计算图的损失或预测张量长期保存在列表中，也会间接保留整条图并占用内存。只做日志记录时可保存 `loss.item()`，需要张量值时可保存 `loss.detach()`。`backward()` 默认释放为反向传播保存的中间值；只有同一图必须反向多次时才使用 `retain_graph=True`，需要让“梯度计算本身”继续参与求导时才使用 `create_graph=True`。二者目的不同。

## 官方教程补充

**对应官方源文件：** `beginner_source/basics/autogradqs_tutorial.py`、`beginner_source/blitz/autograd_tutorial.py`、`beginner_source/understanding_leaf_vs_nonleaf_tutorial.py`

官方材料把 Autograd 描述为运行时构建的有向无环图：叶子张量保存 `.grad`，非叶子节点用 `grad_fn` 记录反向规则。`backward()` 对标量隐式使用上游梯度 1，并把结果累加到叶子梯度；重复使用同一张已释放的图需要重新前向，或有意识地设置 `retain_graph=True`。推理使用 `no_grad`/`inference_mode`，而 `detach` 只切断某个张量之后的图。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

不看上面的说明，尝试回答并验证：

1. 为什么实验 2 必须重新执行前向表达式，而不能直接对实验 1 的同一个 `loss` 再调用 `backward()`？
2. 为什么连续两次反向传播后，`w.grad` 是第一次的两倍？
3. 为什么实验 3 的两个结果都不需要梯度，但 `no_grad` 和 `detach` 不能在所有场景中互换？
4. 在实验 4 后打印 `w2.grad`，预测它的值并解释原因。

如果能够从计算图的生命周期、叶子张量的梯度缓冲区和梯度记录的时机解释这四点，就掌握了本节的核心模型。

## 试一试

1. **改用绝对误差**：把损失替换为 `((x * w).sum() - 4.0).abs()`。先根据 $L=|p-4|$ 预测当前梯度，再运行代码验证。继续尝试让 `p` 恰好等于 4，观察 PyTorch 在不可导点选择的次梯度。每次实验前记得清空 `w.grad`。
2. **有限差分检查**：对每个 $w_i$ 分别计算 $[L(w_i+\varepsilon)-L(w_i-\varepsilon)]/(2\varepsilon)$，例如取 $\varepsilon=10^{-4}$，并与平方误差的 Autograd 梯度比较。有限差分过大时近似误差明显，过小时又可能受浮点舍入影响。
3. **验证 API 差异**：在实验 4 后打印 `w2.grad`；再改用 `loss2.backward()` 重建并运行该实验，对比“返回梯度”和“累积到 `.grad`”两种接口行为。

## 常见错误与调试

- **忘记清梯度**：表现为梯度随迭代意外增大。检查每个训练步骤是否在正确位置调用 `optimizer.zero_grad()`。
- **对非标量输出直接调用 `backward()`**：Autograd 不知道该用怎样的上游梯度。可先用 `sum()` 或 `mean()` 化为标量，或向 `backward(gradient=...)` 传入与输出同形状的向量。
- **原地修改反向传播所需的张量**：可能触发版本计数错误，提示某个变量已被 inplace operation 修改。调试时避免带下划线的原地 API，并可临时启用 `torch.autograd.set_detect_anomaly(True)` 定位问题运算。
- **在训练阶段误用 `no_grad` 或过早 `detach`**：表现为损失没有 `grad_fn`，或报错“does not require grad”。沿损失向前检查各张量的 `requires_grad` 和 `grad_fn`。
- **读取非叶子张量的 `.grad`**：非叶子张量默认不保留 `.grad`。通常应查看叶子参数；若确实需要调试中间梯度，可在反向前调用该张量的 `retain_grad()`。